In [1]:
import numpy as np
import os

def to_hex_8b(val):
    """將 8-bit 整數轉換為 2 碼 Hex 字串 (正確處理二進位補數)"""
    return f"{int(val) & 0xFF:02x}"

def to_hex_32b(val):
    """將 32-bit 整數轉換為 8 碼 Hex 字串 (正確處理二進位補數)"""
    return f"{int(val) & 0xFFFFFFFF:08x}"

def save_matrix_to_hex(matrix, filename, bit_width=8, elements_per_line=16):
    """將矩陣依 Row-Major 順序，每行固定輸出 elements_per_line 個數值到 Hex 檔案"""
    # 自動建立資料夾
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    
    with open(filename, 'w') as f:
        # 將多維矩陣展平成一維
        flat_matrix = matrix.flatten()
        
        # 每次取出 elements_per_line 個元素作為一個 Chunk
        for i in range(0, len(flat_matrix), elements_per_line):
            chunk = flat_matrix[i : i + elements_per_line]
            
            # 如果矩陣大小無法被整除，最後一行補 0 (Padding)
            if len(chunk) < elements_per_line:
                chunk = np.pad(chunk, (0, elements_per_line - len(chunk)), 'constant', constant_values=0)
            
            # 將 Chunk 反轉。確保 chunk[0] 在最右邊 (LSB)
            chunk_flipped = np.flip(chunk)
            
            if bit_width == 8:
                # 轉成 Hex 並拼接寫入檔案 (例如 16 個 INT8 = 32碼字串)
                f.write("".join([to_hex_8b(v) for v in chunk_flipped]) + '\n')
            elif bit_width == 32:
                # 轉成 Hex 並拼接寫入檔案 (例如 4 個 INT32 = 32碼字串)
                f.write("".join([to_hex_32b(v) for v in chunk_flipped]) + '\n')
                
    total_lines = len(flat_matrix) // elements_per_line + (1 if len(flat_matrix) % elements_per_line != 0 else 0)
    print(f"成功導出: {filename} (每行 {elements_per_line} 個 {bit_width}-bit 元素，共 {total_lines} 行)")

In [2]:
def gen_mat_mul_hex(M=16, N=16, K=16, case=0, Identity=0):
    # 設定隨機種子，加入 case 讓不同測資隨機數不同
    np.random.seed(42 + case)

    # 1. 生成資料
    if Identity == 1:
        weight     = 1 * np.identity(16, dtype=np.int8)
        activation = np.random.randint(0, 255, size=(M, K), dtype=np.uint8)
        bias       = np.zeros((N,), dtype=np.int32) # Identity 測試先不加 bias，維持矩陣純淨
    elif Identity == 2:
        weight     = np.random.randint(-128, 127, size=(K, N), dtype=np.int8)
        activation = 1 * np.identity(16, dtype=np.uint8)
        bias       = np.zeros((N,), dtype=np.int32)
    else:
        activation = np.random.randint(0, 255, size=(M, K), dtype=np.uint8)
        weight     = np.random.randint(-128, 127, size=(K, N), dtype=np.int8)
        # 隨機產生 INT32 Bias (模擬硬體累積的量級)
        bias       = np.random.randint(-50000, 50000, size=(N,), dtype=np.int32)
    
    # 2. 執行 GEMM 並加上 Bias
    # np.dot shape: (M, N), bias shape: (N,) -> 自動廣播加到每一列
    opsum = np.dot(activation.astype(np.int32), weight.astype(np.int32)) + bias

    # 3. K-Dimension Tiling 重排
    act_tiled = activation.reshape(M, K // 16, 16).transpose(1, 0, 2)
    weight_tiled = weight.reshape(K // 16, 16, N)

    # 4. 儲存為 Hex 檔案
    # Act & Weight: 8-bit, 每行 4 個 = 32 bit
    save_matrix_to_hex(act_tiled, f"case{case}/act.hex", bit_width=8, elements_per_line=4)
    save_matrix_to_hex(weight_tiled, f"case{case}/weight.hex", bit_width=8, elements_per_line=4)
    
    # 🌟 Bias: 32-bit, 每行 1 個 = 32 bit
    save_matrix_to_hex(bias, f"case{case}/bias.hex", bit_width=32, elements_per_line=1)
    
    # Golden: 維持 32-bit, 每行 1 個 (供 Testbench 逐筆比對)
    save_matrix_to_hex(opsum, f"case{case}/golden.hex", bit_width=32, elements_per_line=1)

## case0
- [16, 16] by [16, 16]
- [act] [Identity] = [act]

In [3]:
gen_mat_mul_hex(M=16, N=16, K=16, case=0, Identity=1)

成功導出: case0/act.hex (每行 4 個 8-bit 元素，共 64 行)
成功導出: case0/weight.hex (每行 4 個 8-bit 元素，共 64 行)
成功導出: case0/bias.hex (每行 1 個 32-bit 元素，共 16 行)
成功導出: case0/golden.hex (每行 1 個 32-bit 元素，共 256 行)


## case1
- [16, 16] by [16, 16]
- [act] [weight] = [psum]
- all random value

In [4]:
gen_mat_mul_hex(16, 16, 16, 1, Identity=0)

成功導出: case1/act.hex (每行 4 個 8-bit 元素，共 64 行)
成功導出: case1/weight.hex (每行 4 個 8-bit 元素，共 64 行)
成功導出: case1/bias.hex (每行 1 個 32-bit 元素，共 16 行)
成功導出: case1/golden.hex (每行 1 個 32-bit 元素，共 256 行)


## case2
- [16, 32] by [32, 16] (2 k-tile accumulate)
- [act] [weight] = [psum]
- all random value

In [5]:
gen_mat_mul_hex(16, 16, K=32, case=2, Identity=0)

成功導出: case2/act.hex (每行 4 個 8-bit 元素，共 128 行)
成功導出: case2/weight.hex (每行 4 個 8-bit 元素，共 128 行)
成功導出: case2/bias.hex (每行 1 個 32-bit 元素，共 16 行)
成功導出: case2/golden.hex (每行 1 個 32-bit 元素，共 256 行)


## case3
- [16, 64] by [64, 16] (4 k-tile accumulate)
- [act] [weight] = [psum]
- all random value

In [6]:
gen_mat_mul_hex(16, 16, K=64, case=3, Identity=0)

成功導出: case3/act.hex (每行 4 個 8-bit 元素，共 256 行)
成功導出: case3/weight.hex (每行 4 個 8-bit 元素，共 256 行)
成功導出: case3/bias.hex (每行 1 個 32-bit 元素，共 16 行)
成功導出: case3/golden.hex (每行 1 個 32-bit 元素，共 256 行)


# case4
- [16, 1536] by [1536, 16] 
- 96 K-tile accumulate, max case in `ViT-small-16`
- random value

In [7]:
gen_mat_mul_hex(16, 16, K=1536, case=4, Identity=0)

成功導出: case4/act.hex (每行 4 個 8-bit 元素，共 6144 行)
成功導出: case4/weight.hex (每行 4 個 8-bit 元素，共 6144 行)
成功導出: case4/bias.hex (每行 1 個 32-bit 元素，共 16 行)
成功導出: case4/golden.hex (每行 1 個 32-bit 元素，共 256 行)
